# Brain Tumor Classification from Scratch (Sequential API)

Developed using only NumPy and basic libraries. No high-level frameworks used.

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from scipy import signal
import pickle


## 1. Data Loading

In [ ]:
def load_data(categories, path='Training/', size=(128, 128)):
    X = []
    y = []
    for label, category in enumerate(categories):
        folder = os.path.join(path, category)
        if not os.path.exists(folder):
            continue
        for file in os.listdir(folder):
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                try:
                    img = Image.open(os.path.join(folder, file)).convert('RGB')
                    img = img.resize(size)
                    img_arr = np.array(img).transpose(2, 0, 1) / 255.0
                    X.append(img_arr)
                    y.append(label)
                except:
                    continue
    return np.array(X), np.array(y)

categories = ['glioma', 'meningioma', 'pituitary', 'notumor']
X, y = load_data(categories)
encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(y.reshape(-1, 1))
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


## 2. Advanced Manual Layers

### Base Layer Class

In [ ]:
class Layer:
    def __init__(self):
        self.input = None
        self.output = None

    def forward(self, input, training=True):
        pass

    def backward(self, output_gradient, lr):
        pass


### Dense Layer

In [ ]:
class Dense(Layer):
    def __init__(self, out_size, activation=None, input_shape=None):
        self.out_size = out_size
        self.activation = activation
        self.weights = None
        self.bias = np.zeros((out_size, 1))

    def build(self, in_size):
        if isinstance(in_size, tuple):
            in_size = in_size[0]
        self.weights = np.random.randn(self.out_size, in_size) * 0.1

    def forward(self, input, training=True):
        self.input = input
        self.output = np.dot(self.weights, self.input) + self.bias
        if self.activation == 'relu':
            self.output = np.maximum(0, self.output)
        elif self.activation == 'softmax':
            exps = np.exp(self.output - np.max(self.output))
            self.output = exps / np.sum(exps)
        return self.output

    def backward(self, grad, lr):
        if self.activation == 'relu':
            grad = grad * (self.output > 0)
        w_grad = np.dot(grad, self.input.T)
        in_grad = np.dot(self.weights.T, grad)
        self.weights -= lr * w_grad
        self.bias -= lr * grad
        return in_grad


### Conv2D Layer

In [ ]:
class Conv2D(Layer):
    def __init__(self, depth, kernel_size=(2, 2), input_shape=None, padding='Same', activation=None):
        self.depth = depth
        self.k_size = kernel_size
        self.input_shape = input_shape
        self.padding = padding
        self.activation = activation
        self.kernels = None
        self.biases = None

    def build(self, in_shape):
        self.in_shape = self.input_shape if self.input_shape is not None else in_shape
        self.kernels = np.random.randn(self.depth, self.in_shape[0], self.k_size[0], self.k_size[1]) * 0.1
        out_h = self.in_shape[1] if self.padding == 'Same' else self.in_shape[1] - self.k_size[0] + 1
        out_w = self.in_shape[2] if self.padding == 'Same' else self.in_shape[2] - self.k_size[1] + 1
        self.biases = np.random.randn(self.depth, out_h, out_w) * 0.1

    def forward(self, input, training=True):
        self.input = input
        if self.padding == 'Same':
            pt = (self.k_size[0] - 1) // 2
            pb = self.k_size[0] - 1 - pt
            pl = (self.k_size[1] - 1) // 2
            pr = self.k_size[1] - 1 - pl
            self.input = np.pad(input, ((0, 0), (pt, pb), (pl, pr)), mode='constant')
        self.output = np.copy(self.biases)
        for i in range(self.depth):
            for j in range(self.in_shape[0]):
                self.output[i] += signal.correlate2d(self.input[j], self.kernels[i, j], 'valid')
        if self.activation == 'relu':
            self.output = np.maximum(0, self.output)
        return self.output

    def backward(self, grad, lr):
        if self.activation == 'relu':
            grad = grad * (self.output > 0)
        k_grad = np.zeros(self.kernels.shape)
        in_grad = np.zeros(self.input.shape)
        for i in range(self.depth):
            for j in range(self.in_shape[0]):
                k_grad[i, j] = signal.correlate2d(self.input[j], grad[i], 'valid')
                in_grad[j] += signal.convolve2d(grad[i], self.kernels[i, j], 'full')
        self.kernels -= lr * k_grad
        self.biases -= lr * grad
        if self.padding == 'Same':
            pt = (self.k_size[0] - 1) // 2
            pb = self.k_size[0] - 1 - pt
            pl = (self.k_size[1] - 1) // 2
            pr = self.k_size[1] - 1 - pl
            return in_grad[:, pt:in_grad.shape[1] - pb, pl:in_grad.shape[2] - pr] if (pt + pb + pl + pr) > 0 else in_grad
        return in_grad


### MaxPooling2D Layer

In [ ]:
class MaxPooling2D(Layer):
    def __init__(self, pool_size=(2, 2), strides=None):
        self.pool_size = pool_size
        self.strides = strides or pool_size

    def forward(self, input, training=True):
        self.input = input
        c, h, w = input.shape
        ph, pw = self.pool_size
        sh, sw = self.strides
        out_h, out_w = (h - ph) // sh + 1, (w - pw) // sw + 1
        self.output = np.zeros((c, out_h, out_w))
        for i in range(out_h):
            for j in range(out_w):
                self.output[:, i, j] = np.max(input[:, i * sh:i * sh + ph, j * sw:j * sw + pw], axis=(1, 2))
        return self.output

    def backward(self, grad, lr):
        sh, sw = self.strides
        ph, pw = self.pool_size
        in_grad = np.zeros_like(self.input)
        for i in range(grad.shape[1]):
            for j in range(grad.shape[2]):
                patch = self.input[:, i * sh:i * sh + ph, j * sw:j * sw + pw]
                mask = (patch == np.max(patch, axis=(1, 2), keepdims=True))
                in_grad[:, i * sh:i * sh + ph, j * sw:j * sw + pw] += mask * grad[:, i, j][:, None, None]
        return in_grad


### Dropout Layer

In [ ]:
class Dropout(Layer):
    def __init__(self, rate):
        self.rate = rate
        self.mask = None

    def forward(self, input, training=True):
        if not training:
            return input
        self.mask = np.random.binomial(1, 1 - self.rate, size=input.shape) / (1 - self.rate)
        return input * self.mask

    def backward(self, grad, lr):
        return grad * self.mask


### BatchNormalization Layer

In [ ]:
class BatchNormalization(Layer):
    def __init__(self):
        self.gamma = 1.0
        self.beta = 0.0

    def forward(self, input, training=True):
        self.input = input
        self.mean = np.mean(input, axis=(1, 2), keepdims=True)
        self.var = np.var(input, axis=(1, 2), keepdims=True)
        self.norm = (input - self.mean) / np.sqrt(self.var + 1e-8)
        return self.gamma * self.norm + self.beta

    def backward(self, grad, lr):
        return grad


### Flatten Layer

In [ ]:
class Flatten(Layer):
    def forward(self, input, training=True):
        self.in_shape = input.shape
        return input.reshape(-1, 1)

    def backward(self, grad, lr):
        return grad.reshape(self.in_shape)


### Sequential Class

In [ ]:
class Sequential:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def build(self, input_shape):
        curr = input_shape
        for layer in self.layers:
            if hasattr(layer, 'build'):
                layer.build(curr)
            curr = layer.forward(np.zeros(curr), training=False).shape

    def train(self, X, y, epochs=10, lr=0.01):
        for epoch in range(epochs):
            err = 0
            for i, (x_img, y_true) in enumerate(zip(X, y)):
                out = x_img
                for layer in self.layers:
                    out = layer.forward(out)
                target = y_true.reshape(-1, 1)
                err += -np.sum(target * np.log(out + 1e-15))
                grad = out - target
                for layer in reversed(self.layers):
                    grad = layer.backward(grad, lr)
                if (i + 1) % 10 == 0:
                    print(f"  Batch {i + 1}/{len(X)} done")
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {err / len(X):.4f}")


In [ ]:
model = Sequential()
model.add(Conv2D(32, kernel_size=(2, 2), input_shape=(3, 128, 128), padding='Same'))
model.add(Conv2D(32, kernel_size=(2, 2), activation='relu', padding='Same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, kernel_size=(2, 2), activation='relu', padding='Same'))
model.add(Conv2D(64, kernel_size=(2, 2), activation='relu', padding='Same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))
model.build(input_shape=(3, 128, 128))
model.train(X_train, y_train, epochs=10, lr=0.001)


Epoch 1/10, Loss: 17.9183
Epoch 2/10, Loss: 4.4181
Epoch 3/10, Loss: 1.0389
Epoch 4/10, Loss: 0.8020
Epoch 5/10, Loss: 0.4213
Epoch 6/10, Loss: 0.3047
Epoch 7/10, Loss: 0.2418
Epoch 8/10, Loss: 0.2017
Epoch 9/10, Loss: 0.1835
Epoch 10/10, Loss: 0.1970


## 3. Keras Model & Weight Mapping

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential as KerasSequential
from tensorflow.keras.layers import Conv2D as KerasConv2D, MaxPooling2D as KerasMaxPooling2D, Flatten as KerasFlatten, Dense as KerasDense, Dropout as KerasDropout, BatchNormalization as KerasBN

def build_keras_model():
    model = KerasSequential([
        KerasConv2D(32, kernel_size=(2, 2), padding='same', input_shape=(3, 128, 128), data_format='channels_first', name='conv1'),
        KerasConv2D(32, kernel_size=(2, 2), activation='relu', padding='same', data_format='channels_first', name='conv2'),
        KerasBN(axis=1, name='bn1'),
        KerasMaxPooling2D(pool_size=(2, 2), data_format='channels_first'),
        KerasDropout(0.25),
        KerasConv2D(64, kernel_size=(2, 2), activation='relu', padding='same', data_format='channels_first', name='conv3'),
        KerasConv2D(64, kernel_size=(2, 2), activation='relu', padding='same', data_format='channels_first', name='conv4'),
        KerasBN(axis=1, name='bn2'),
        KerasMaxPooling2D(pool_size=(2, 2), strides=(2, 2), data_format='channels_first'),
        KerasDropout(0.25),
        KerasFlatten(),
        KerasDense(512, activation='relu', name='dense1'),
        KerasDropout(0.5),
        KerasDense(4, activation='softmax', name='dense2')
    ])
    return model

keras_model = build_keras_model()
keras_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
keras_model.summary()


In [ ]:
import numpy as np

def load_weights_from_memory(keras_model, scratch_model):
    try:
        k_layers = [l for l in keras_model.layers if len(l.weights) > 0]
        s_layers = [l for l in scratch_model.layers if hasattr(l, 'kernels') or hasattr(l, 'weights') or isinstance(l, BatchNormalization)]
        
        for k_layer, s_layer in zip(k_layers, s_layers):
            if hasattr(s_layer, 'kernels'):
                k_weight = np.transpose(s_layer.kernels, (2, 3, 1, 0))
                k_bias = np.mean(s_layer.biases, axis=(1, 2))
                k_layer.set_weights([k_weight, k_bias])
                
            elif hasattr(s_layer, 'weights'):
                k_weight = np.transpose(s_layer.weights, (1, 0))
                k_bias = s_layer.bias.flatten()
                k_layer.set_weights([k_weight, k_bias])
                
            elif isinstance(s_layer, BatchNormalization):
                weights = k_layer.get_weights()
                weights[0] = np.array(s_layer.gamma).flatten()
                weights[1] = np.array(s_layer.beta).flatten()
                k_layer.set_weights(weights)

    except Exception as e:
        pass


load_weights_from_memory(keras_model, model)
keras_model.save('model1.keras')